# VinDatathon 2026 - Task 3: Sales Forecasting

Calendar-only forecasting pipeline for Datathon 2026 Round 1 (VinTelligence / VinUniversity).

**Task**: Predict daily `Revenue` and `COGS` for 548 days (2023-01-01 to 2024-07-01) using training data from `sales.csv` (2012-07-04 to 2022-12-31).

**Pipeline**: Ridge + LightGBM (with Q-specialists) + Prophet → ensemble blend → calibration → submission.

**Validation**: 3-fold time-series CV (Fold A/B/C) with auto-calibration from Fold A.

**Metrics**: MAE, RMSE, R² (evaluated on Kaggle leaderboard).

## 1. Imports and Configuration

Prophet is required by design. If this cell fails, install dependencies from `requirements.txt` before running the notebook.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import lightgbm as lgb

try:
    from prophet import Prophet
except ImportError as exc:
    raise ImportError(
        "Prophet is required for this notebook. Install it with: pip install prophet"
    ) from exc

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)

SEED = 42
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name != "USHIET-DATATHON26":
    ROOT = ROOT / "USHIET-DATATHON26"

DATA_DIR = ROOT / "data"
SUBMISSION_DIR = ROOT / "submission"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "sales.csv"
SAMPLE_PATH = DATA_DIR / "sample_submission.csv"
OUT_PATH = SUBMISSION_DIR / "submission.csv"

TARGETS = ["Revenue", "COGS"]
ALPHA_DEFAULT = 0.60
MODEL_WEIGHTS_DEFAULT = {"ridge": 0.10, "prophet": 0.10, "lgb": 0.80}
QBOOST = 2.0

LGB_PARAMS = dict(
    objective="regression",
    metric="mae",
    learning_rate=0.03,
    num_leaves=63,
    min_data_in_leaf=30,
    feature_fraction=0.85,
    bagging_fraction=0.85,
    bagging_freq=5,
    lambda_l2=1.0,
    seed=SEED,
    verbosity=-1,
)

print("Root:", ROOT)
print("Output:", OUT_PATH)

## 2. Load Data

In [2]:
sales = pd.read_csv(TRAIN_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
sample_submission = pd.read_csv(SAMPLE_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

assert sales["Date"].is_monotonic_increasing
assert sample_submission["Date"].is_monotonic_increasing
assert list(sample_submission.columns) == ["Date", "Revenue", "COGS"]

print("sales:", sales.shape, sales["Date"].min().date(), "->", sales["Date"].max().date())
print("sample:", sample_submission.shape, sample_submission["Date"].min().date(), "->", sample_submission["Date"].max().date())
display(sales.head())
display(sample_submission.head())

sales: (3833, 3) 2012-07-04 -> 2022-12-31
sample: (548, 3) 2023-01-01 -> 2024-07-01


,Date,Revenue,COGS
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79


,Date,Revenue,COGS
0,2023-01-01,2665507.20,2518885.15
1,2023-01-02,1280007.89,1136463.00
2,2023-01-03,1015899.51,822721.12
3,2023-01-04,1142997.27,914554.18
4,2023-01-05,1236312.34,984390.24


## 3. Calendar-Only Feature Engineering

In [3]:
PROMO_SCHEDULE = [
    ("spring_sale", 3, 18, 30, 12, True),
    ("mid_year", 6, 23, 29, 18, True),
    ("fall_launch", 8, 30, 32, 10, True),
    ("year_end", 11, 18, 45, 20, True),
    ("urban_blowout", 7, 30, 33, None, "odd"),
    ("rural_special", 1, 30, 30, 15, "odd"),
]

VN_FIXED_HOLIDAYS = [
    (1, 1, "new_year"),
    (3, 8, "womens_day"),
    (4, 30, "reunification"),
    (5, 1, "labor_day"),
    (9, 2, "national_day"),
    (10, 20, "vn_womens_day"),
    (11, 11, "dd_1111"),
    (12, 12, "dd_1212"),
    (12, 24, "christmas_eve"),
    (12, 25, "christmas"),
]

TET_DATES = {
    2012: "2012-01-23",
    2013: "2013-02-10",
    2014: "2014-01-31",
    2015: "2015-02-19",
    2016: "2016-02-08",
    2017: "2017-01-28",
    2018: "2018-02-16",
    2019: "2019-02-05",
    2020: "2020-01-25",
    2021: "2021-02-12",
    2022: "2022-02-01",
    2023: "2023-01-22",
    2024: "2024-02-10",
}


def build_features(dates: pd.Series | pd.DatetimeIndex) -> pd.DataFrame:
    dates = pd.to_datetime(pd.Series(dates)).reset_index(drop=True)
    df = pd.DataFrame({"Date": dates})
    d = df["Date"]

    df["year"] = d.dt.year
    df["month"] = d.dt.month
    df["day"] = d.dt.day
    df["dow"] = d.dt.dayofweek
    df["doy"] = d.dt.dayofyear
    df["quarter"] = d.dt.quarter
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    df["days_to_eom"] = d.dt.days_in_month - df["day"]
    df["days_from_som"] = df["day"] - 1
    df["dim"] = d.dt.days_in_month

    for k in [1, 2, 3]:
        df[f"is_last{k}"] = (df["days_to_eom"] <= k - 1).astype(int)
        df[f"is_first{k}"] = (df["days_from_som"] <= k - 1).astype(int)

    df["t_days"] = (d - pd.Timestamp("2020-01-01")).dt.days
    df["t_years"] = df["t_days"] / 365.25
    df["regime_pre2019"] = (df["year"] <= 2018).astype(int)
    df["regime_2019"] = (df["year"] == 2019).astype(int)
    df["regime_post2019"] = (df["year"] >= 2020).astype(int)

    tau = 2 * np.pi
    for k in [1, 2, 3, 4, 5]:
        df[f"sin_y{k}"] = np.sin(tau * k * df["doy"] / 365.25)
        df[f"cos_y{k}"] = np.cos(tau * k * df["doy"] / 365.25)
    for k in [1, 2]:
        df[f"sin_w{k}"] = np.sin(tau * k * df["dow"] / 7.0)
        df[f"cos_w{k}"] = np.cos(tau * k * df["dow"] / 7.0)
    for k in [1, 2]:
        df[f"sin_m{k}"] = np.sin(tau * k * (df["day"] - 1) / df["dim"])
        df[f"cos_m{k}"] = np.cos(tau * k * (df["day"] - 1) / df["dim"])

    for month, day, name in VN_FIXED_HOLIDAYS:
        df[f"hol_{name}"] = ((df["month"] == month) & (df["day"] == day)).astype(int)

    tet_lut = {year: pd.Timestamp(value) for year, value in TET_DATES.items()}

    def nearest_tet_diff(day_value: pd.Timestamp) -> int:
        candidates = [tet_lut.get(day_value.year - 1), tet_lut.get(day_value.year), tet_lut.get(day_value.year + 1)]
        diffs = [(day_value - c).days for c in candidates if c is not None]
        diffs = [x for x in diffs if abs(x) <= 45]
        return min(diffs, key=abs) if diffs else 999

    tet_diff = np.array([nearest_tet_diff(x) for x in d])
    df["tet_days_diff"] = tet_diff
    df["tet_in_7"] = (np.abs(tet_diff) <= 7).astype(int)
    df["tet_in_14"] = (np.abs(tet_diff) <= 14).astype(int)
    df["tet_before_7"] = ((tet_diff >= -7) & (tet_diff < 0)).astype(int)
    df["tet_after_7"] = ((tet_diff > 0) & (tet_diff <= 7)).astype(int)
    df["tet_on"] = (tet_diff == 0).astype(int)

    def is_black_friday(day_value: pd.Timestamp) -> int:
        if day_value.month != 11:
            return 0
        last = pd.Timestamp(year=day_value.year, month=11, day=30)
        last_friday = last - pd.Timedelta(days=(last.dayofweek - 4) % 7)
        return int(day_value == last_friday)

    df["hol_black_friday"] = [is_black_friday(x) for x in d]

    years = range(int(df["year"].min()) - 1, int(df["year"].max()) + 2)
    for name, start_month, start_day, duration, discount, recurrence in PROMO_SCHEDULE:
        in_promo = np.zeros(len(df), dtype=int)
        since = np.full(len(df), -1.0)
        until = np.full(len(df), -1.0)
        discount_value = np.zeros(len(df), dtype=float)

        for year in years:
            if recurrence == "odd" and year % 2 == 0:
                continue
            start = pd.Timestamp(year=year, month=start_month, day=start_day)
            end = start + pd.Timedelta(days=duration)
            mask = (d >= start) & (d <= end)
            in_promo[mask] = 1
            since[mask] = (d[mask] - start).dt.days
            until[mask] = (end - d[mask]).dt.days
            discount_value[mask] = discount or 0

        df[f"promo_{name}"] = in_promo
        df[f"promo_{name}_since"] = since
        df[f"promo_{name}_until"] = until
        df[f"promo_{name}_disc"] = discount_value

    df["is_odd_year"] = (df["year"] % 2).astype(int)
    return df


feature_preview = build_features(sample_submission["Date"].head(10))
print("Feature preview shape:", feature_preview.shape)
display(feature_preview.head())

Feature preview shape: (10, 82)


,Date,year,month,day,dow,doy,quarter,is_weekend,days_to_eom,days_from_som,dim,is_last1,is_first1,is_last2,is_first2,is_last3,is_first3,t_days,t_years,regime_pre2019,regime_2019,regime_post2019,sin_y1,cos_y1,sin_y2,cos_y2,sin_y3,cos_y3,sin_y4,cos_y4,sin_y5,cos_y5,sin_w1,cos_w1,sin_w2,cos_w2,sin_m1,cos_m1,sin_m2,cos_m2,hol_new_year,hol_womens_day,hol_reunification,hol_labor_day,hol_national_day,hol_vn_womens_day,hol_dd_1111,hol_dd_1212,hol_christmas_eve,hol_christmas,tet_days_diff,tet_in_7,tet_in_14,tet_before_7,tet_after_7,tet_on,hol_black_friday,promo_spring_sale,promo_spring_sale_since,promo_spring_sale_until,promo_spring_sale_disc,promo_mid_year,promo_mid_year_since,promo_mid_year_until,promo_mid_year_disc,promo_fall_launch,promo_fall_launch_since,promo_fall_launch_until,promo_fall_launch_disc,promo_year_end,promo_year_end_since,promo_year_end_until,promo_year_end_disc,promo_urban_blowout,promo_urban_blowout_since,promo_urban_blowout_until,promo_urban_blowout_disc,promo_rural_special,promo_rural_special_since,promo_rural_special_until,promo_rural_special_disc,is_odd_year
0,2023-01-01,2023,1,1,6,1,1,1,30,0,31,0,1,0,1,0,1,1096,3.000684,0,0,1,0.017202,0.999852,0.034398,0.999408,0.051584,0.998669,0.068755,0.997634,0.085906,0.996303,-0.781831,0.623490,-0.974928,-0.222521,0.000000,1.000000,0.000000,1.000000,1,0,0,0,0,0,0,0,0,0,-21,0,0,0,0,0,0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,1,44.0,1.0,20.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,1
1,2023-01-02,2023,1,2,0,2,1,0,29,1,31,0,0,0,1,0,1,1097,3.003422,0,0,1,0.034398,0.999408,0.068755,0.997634,0.103031,0.994678,0.137185,0.990545,0.171177,0.985240,0.000000,1.000000,0.000000,1.000000,0.201299,0.979530,0.394356,0.918958,0,0,0,0,0,0,0,0,0,0,-20,0,0,0,0,0,0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,1,45.0,0.0,20.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,1
2,2023-01-03,2023,1,3,1,3,1,0,28,2,31,0,0,0,0,0,1,1098,3.006160,0,0,1,0.051584,0.998669,0.103031,0.994678,0.154204,0.988039,0.204966,0.978769,0.255182,0.966893,0.781831,0.623490,0.974928,-0.222521,0.394356,0.918958,0.724793,0.688967,0,0,0,0,0,0,0,0,0,0,-19,0,0,0,0,0,0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,1
3,2023-01-04,2023,1,4,2,4,1,0,27,3,31,0,0,0,0,0,0,1099,3.008898,0,0,1,0.068755,0.997634,0.137185,0.990545,0.204966,0.978769,0.271777,0.962360,0.337301,0.941397,0.974928,-0.222521,-0.433884,-0.900969,0.571268,0.820763,0.937752,0.347305,0,0,0,0,0,0,0,0,0,0,-18,0,0,0,0,0,0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,1
4,2023-01-05,2023,1,5,3,5,1,0,26,4,31,0,0,0,0,0,0,1100,3.011636,0,0,1,0.085906,0.996303,0.171177,0.985240,0.255182,0.966893,0.337301,0.941397,0.416926,0.908940,0.433884,-0.900969,-0.781831,0.623490,0.724793,0.688967,0.998717,-0.050649,0,0,0,0,0,0,0,0,0,0,-17,0,0,0,0,0,0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,0,-1.0,-1.0,0.0,1


## 4. Model Utilities

In [4]:
def get_feature_columns(feature_df: pd.DataFrame) -> list[str]:
    return [c for c in feature_df.columns if c != "Date"]


def high_era_weights(dates: pd.Series, target_quarter: int | None = None, qboost: float = QBOOST) -> np.ndarray:
    dates = pd.to_datetime(dates)
    years = dates.dt.year.values
    weights = np.full(len(dates), 0.01, dtype=float)
    weights[(years >= 2014) & (years <= 2018)] = 1.0
    if target_quarter is not None:
        quarters = dates.dt.quarter.values
        weights[quarters == target_quarter] *= qboost
    return weights


def train_ridge_model(X_train: pd.DataFrame, y_log: np.ndarray, alpha: float = 3.0):
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0).replace(0, 1)
    model = Ridge(alpha=alpha, random_state=SEED)
    model.fit((X_train - mu) / sigma, y_log)
    return model, mu, sigma


def predict_ridge_model(model_bundle, X_test: pd.DataFrame) -> np.ndarray:
    model, mu, sigma = model_bundle
    return np.exp(model.predict((X_test - mu) / sigma))


def train_lgb_model(X_train: pd.DataFrame, y_log: np.ndarray, dates: pd.Series, weights: np.ndarray):
    intern_cutoff = pd.to_datetime(dates).max() - pd.Timedelta(days=180)
    fit_idx = (pd.to_datetime(dates) <= intern_cutoff).values
    val_idx = ~fit_idx

    if val_idx.sum() < 30:
        dataset = lgb.Dataset(X_train, y_log, weight=weights)
        return lgb.train(LGB_PARAMS, dataset, num_boost_round=1200)

    booster_es = lgb.train(
        LGB_PARAMS,
        lgb.Dataset(X_train.loc[fit_idx], y_log[fit_idx], weight=weights[fit_idx]),
        num_boost_round=5000,
        valid_sets=[lgb.Dataset(X_train.loc[val_idx], y_log[val_idx])],
        callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(0)],
    )
    best_iter = booster_es.best_iteration or 1200
    return lgb.train(
        LGB_PARAMS,
        lgb.Dataset(X_train, y_log, weight=weights),
        num_boost_round=best_iter,
    )


def predict_lgb_model(model, X_test: pd.DataFrame) -> np.ndarray:
    return np.exp(model.predict(X_test))


def prophet_regressor_columns(feature_cols: list[str]) -> list[str]:
    prefixes = ("promo_", "hol_", "tet_", "is_", "sin_", "cos_")
    return [c for c in feature_cols if c.startswith(prefixes)]


def train_prophet_model(train_frame: pd.DataFrame, feature_cols: list[str], target: str):
    reg_cols = prophet_regressor_columns(feature_cols)
    df = train_frame[["Date", target] + reg_cols].copy()
    df = df[df["Date"] >= "2020-01-01"].copy()
    if len(df) < 365:
        df = train_frame[["Date", target] + reg_cols].copy()
    df = df.rename(columns={"Date": "ds", target: "y"})
    df["y"] = np.log(df["y"].clip(lower=1))

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode="multiplicative",
        changepoint_prior_scale=0.05,
    )
    for col in reg_cols:
        model.add_regressor(col)
    model.fit(df)
    return model, reg_cols


def predict_prophet_model(model_bundle, feature_frame: pd.DataFrame) -> np.ndarray:
    model, reg_cols = model_bundle
    future = feature_frame[["Date"] + reg_cols].rename(columns={"Date": "ds"})
    pred = model.predict(future)["yhat"].values
    return np.exp(pred)


def score_predictions(y_true: pd.DataFrame, y_pred: pd.DataFrame, label: str) -> dict:
    rows = []
    for target in TARGETS:
        rows.append({
            "split": label,
            "target": target,
            "MAE": mean_absolute_error(y_true[target], y_pred[target]),
            "RMSE": root_mean_squared_error(y_true[target], y_pred[target]),
            "R2": r2_score(y_true[target], y_pred[target]),
        })
    avg = pd.DataFrame(rows)[["MAE", "RMSE", "R2"]].mean().to_dict()
    rows.append({"split": label, "target": "average", **avg})
    return rows

## 5. Pipeline Components

In [5]:
def fit_target_components(train_df: pd.DataFrame, feature_cols: list[str], target: str):
    X_train = train_df[feature_cols]
    y_log = np.log(train_df[target].clip(lower=1).values)
    dates = train_df["Date"]

    ridge = train_ridge_model(X_train, y_log)
    lgb_base = train_lgb_model(X_train, y_log, dates, high_era_weights(dates))
    prophet = train_prophet_model(train_df, feature_cols, target)

    specialists = {}
    for quarter in [1, 2, 3, 4]:
        weights = high_era_weights(dates, target_quarter=quarter, qboost=QBOOST)
        specialists[quarter] = train_lgb_model(X_train, y_log, dates, weights)

    return {
        "ridge": ridge,
        "lgb_base": lgb_base,
        "prophet": prophet,
        "specialists": specialists,
    }


def predict_target_components(models: dict, test_features: pd.DataFrame, feature_cols: list[str]) -> dict[str, np.ndarray]:
    X_test = test_features[feature_cols]
    quarters = test_features["Date"].dt.quarter.values

    spec_composed = np.zeros(len(test_features), dtype=float)
    for quarter, specialist in models["specialists"].items():
        mask = quarters == quarter
        if mask.any():
            spec_composed[mask] = predict_lgb_model(specialist, X_test.loc[mask])

    return {
        "ridge": predict_ridge_model(models["ridge"], X_test),
        "lgb_base": predict_lgb_model(models["lgb_base"], X_test),
        "prophet": predict_prophet_model(models["prophet"], test_features),
        "lgb_spec": spec_composed,
    }


def blend_target_predictions(
    components: dict[str, np.ndarray],
    alpha: float = ALPHA_DEFAULT,
    weights: dict[str, float] = MODEL_WEIGHTS_DEFAULT,
    calibration: float = 1.0,
) -> np.ndarray:
    lgb_blend = alpha * components["lgb_spec"] + (1 - alpha) * components["lgb_base"]
    raw = (
        weights["ridge"] * components["ridge"]
        + weights["prophet"] * components["prophet"]
        + weights["lgb"] * lgb_blend
    )
    return np.maximum(raw * calibration, 0)


def fit_predict_components(train_raw: pd.DataFrame, predict_dates: pd.Series):
    train_features = build_features(train_raw["Date"])
    train_frame = pd.concat([train_features, train_raw[TARGETS].reset_index(drop=True)], axis=1)
    test_features = build_features(predict_dates)
    feature_cols = get_feature_columns(train_features)

    all_components = {}
    fitted_models = {}
    for target in TARGETS:
        print(f"Training components for {target} on {len(train_frame):,} rows...")
        fitted_models[target] = fit_target_components(train_frame, feature_cols, target)
        all_components[target] = predict_target_components(fitted_models[target], test_features, feature_cols)

    return all_components, fitted_models, feature_cols, test_features

## 6. Three-Fold Time-Series CV

Three folds per the training document:
- **Fold A (primary)**: train ≤ 2021-12, validate 2022 — closest distribution to test; used for ablation and calibration
- **Fold B (stability)**: train ≤ 2020-12, validate 2021 — checks year-over-year stability
- **Fold C (horizon)**: train ≤ 2021-06, validate 2021-07 → 2022-06 — mimics 12-month continuous horizon

In [ ]:
def make_threefold_splits(df: pd.DataFrame) -> list[dict]:
    fold_specs = [
        ("Fold_A", "2021-12-31", "2022-01-01", "2022-12-31"),
        ("Fold_B", "2020-12-31", "2021-01-01", "2021-12-31"),
        ("Fold_C", "2021-06-30", "2021-07-01", "2022-06-30"),
    ]
    splits = []
    for name, te, vs, ve in fold_specs:
        train_end, valid_start, valid_end = pd.Timestamp(te), pd.Timestamp(vs), pd.Timestamp(ve)
        splits.append({
            "name": name,
            "train_end": train_end,
            "valid_start": valid_start,
            "valid_end": valid_end,
            "train_mask": df["Date"] <= train_end,
            "valid_mask": (df["Date"] >= valid_start) & (df["Date"] <= valid_end),
        })
    return splits


cv_splits = make_threefold_splits(sales)
pd.DataFrame([
    {
        "fold": s["name"],
        "train_end": s["train_end"].date(),
        "valid_start": s["valid_start"].date(),
        "valid_end": s["valid_end"].date(),
        "train_rows": int(s["train_mask"].sum()),
        "valid_rows": int(s["valid_mask"].sum()),
    }
    for s in cv_splits
])

In [ ]:
def evaluate_blend_grid(component_store: list[dict], alpha_grid=None, model_weight_grid=None) -> pd.DataFrame:
    if alpha_grid is None:
        alpha_grid = [0.50, 0.55, 0.60, 0.65, 0.70]
    if model_weight_grid is None:
        model_weight_grid = [
            {"ridge": 0.10, "prophet": 0.10, "lgb": 0.80},
            {"ridge": 0.05, "prophet": 0.10, "lgb": 0.85},
            {"ridge": 0.10, "prophet": 0.05, "lgb": 0.85},
        ]

    rows = []
    for alpha in alpha_grid:
        for weights in model_weight_grid:
            fold_maes = []
            fold_a_mae = None
            for item in component_store:
                pred = pd.DataFrame({"Date": item["valid"]["Date"].values})
                for target in TARGETS:
                    pred[target] = blend_target_predictions(
                        item["components"][target], alpha=alpha, weights=weights, calibration=1.0,
                    )
                mae = np.mean([mean_absolute_error(item["valid"][t], pred[t]) for t in TARGETS])
                fold_maes.append(mae)
                if item["split"] == "Fold_A":
                    fold_a_mae = mae
            rows.append({
                "alpha": alpha,
                "w_ridge": weights["ridge"],
                "w_prophet": weights["prophet"],
                "w_lgb": weights["lgb"],
                "mean_mae": float(np.mean(fold_maes)),
                "fold_a_mae": float(fold_a_mae) if fold_a_mae else float(np.mean(fold_maes)),
            })
    return pd.DataFrame(rows).sort_values(["fold_a_mae", "mean_mae"]).reset_index(drop=True)


def compute_calibration(component_store: list[dict], alpha: float, weights: dict) -> dict[str, float]:
    fold_a = [item for item in component_store if item["split"] == "Fold_A"]
    if not fold_a:
        return {t: 1.0 for t in TARGETS}
    item = fold_a[0]
    cal = {}
    for target in TARGETS:
        pred = blend_target_predictions(item["components"][target], alpha=alpha, weights=weights, calibration=1.0)
        cal[target] = float(item["valid"][target].mean() / pred.mean())
    return cal


# --- Run 3-fold CV ---
cv_component_store = []
cv_metric_rows = []

for split in cv_splits:
    print("\n" + "=" * 80)
    print(f"{split['name']}: train <= {split['train_end'].date()}, valid {split['valid_start'].date()} -> {split['valid_end'].date()}")
    train_part = sales.loc[split["train_mask"]].reset_index(drop=True)
    valid_part = sales.loc[split["valid_mask"]].reset_index(drop=True)

    components, _, _, _ = fit_predict_components(train_part, valid_part["Date"])
    cv_component_store.append({"split": split["name"], "valid": valid_part, "components": components})

    pred_default = pd.DataFrame({"Date": valid_part["Date"]})
    for target in TARGETS:
        pred_default[target] = blend_target_predictions(
            components[target], alpha=ALPHA_DEFAULT, weights=MODEL_WEIGHTS_DEFAULT, calibration=1.0,
        )
    cv_metric_rows.extend(score_predictions(valid_part[TARGETS], pred_default[TARGETS], split["name"]))

cv_metrics = pd.DataFrame(cv_metric_rows)
display(cv_metrics)

# --- Find best blend ---
blend_results = evaluate_blend_grid(cv_component_store)
display(blend_results.head(10))

best = blend_results.iloc[0]
BEST_ALPHA = float(best["alpha"])
BEST_MODEL_WEIGHTS = {"ridge": float(best["w_ridge"]), "prophet": float(best["w_prophet"]), "lgb": float(best["w_lgb"])}

# --- Auto-calibration from Fold A ---
CALIBRATION = compute_calibration(cv_component_store, BEST_ALPHA, BEST_MODEL_WEIGHTS)

print(f"\nBest alpha: {BEST_ALPHA}")
print(f"Best model weights: {BEST_MODEL_WEIGHTS}")
print(f"Auto-calibration: {CALIBRATION}")

## 7. Final Training and Submission

In [ ]:
final_components, final_models, final_feature_cols, final_test_features = fit_predict_components(
    sales,
    sample_submission["Date"],
)

submission = pd.DataFrame({"Date": sample_submission["Date"]})
for target in TARGETS:
    submission[target] = blend_target_predictions(
        final_components[target],
        alpha=BEST_ALPHA,
        weights=BEST_MODEL_WEIGHTS,
        calibration=CALIBRATION[target],
    ).round(2)

assert len(submission) == len(sample_submission)
assert list(submission.columns) == ["Date", "Revenue", "COGS"]
assert submission["Date"].equals(sample_submission["Date"])
assert not submission[TARGETS].isna().any().any()
assert (submission[TARGETS] >= 0).all().all()

submission_to_save = submission.copy()
submission_to_save["Date"] = submission_to_save["Date"].dt.strftime("%Y-%m-%d")
submission_to_save.to_csv(OUT_PATH, index=False)

print(f"Saved {len(submission_to_save):,} rows to {OUT_PATH}")
display(submission_to_save.head(10))
display(submission_to_save.tail(10))

## 8. Explainability: LightGBM Feature Importance

In [ ]:
importance_frames = []
for target in TARGETS:
    booster = final_models[target]["lgb_base"]
    imp = pd.DataFrame({
        "target": target,
        "feature": final_feature_cols,
        "importance_gain": booster.feature_importance(importance_type="gain"),
        "importance_split": booster.feature_importance(importance_type="split"),
    }).sort_values("importance_gain", ascending=False)
    importance_frames.append(imp)
    print(f"Top features for {target}")
    display(imp.head(25))

feature_importance = pd.concat(importance_frames, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, target in zip(axes, TARGETS):
    plot_df = feature_importance[feature_importance["target"] == target].head(20).iloc[::-1]
    ax.barh(plot_df["feature"], plot_df["importance_gain"])
    ax.set_title(f"LightGBM gain importance - {target}")
    ax.set_xlabel("Gain")
plt.tight_layout()
plt.show()